<a href="https://colab.research.google.com/github/Aadish2423/Speech-Emotion-Recognition-/blob/main/speech_emotion_recognition(ravdess).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets

In [2]:
import opendatasets as od


In [3]:

od.download("https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio")


Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio


100%|██████████| 429M/429M [00:05<00:00, 77.5MB/s]


In [4]:
import os
import pandas as pd
import kagglehub


def create_ravdess_dataset(directory_path):
    emotion_map = {
        '01': 'neutral',
        '02': 'calm',
        '03': 'happy',
        '04': 'sad',
        '05': 'angry',
        '06': 'fearful',
        '07': 'disgust',
        '08': 'surprised'
    }
    dataset = []
    for subdir, _, files in os.walk(directory_path):
        for file in files:
            if file.endswith('.wav'):
                parts = file.split('-')
                if len(parts) > 2:
                    emo_code = parts[2]
                    if emo_code in emotion_map:
                        emotion = emotion_map[emo_code]
                        full_path = os.path.join(subdir, file)
                        dataset.append({'path': full_path, 'emotion': emotion})
    return dataset

# Use the correct path for extraction
ravdess_path = './ravdess-emotional-speech-audio' # Define ravdess_path
ravdess_data = create_ravdess_dataset(ravdess_path)
df_ravdess = pd.DataFrame(ravdess_data)

# Save and display all data
df_ravdess.to_csv('ravdess.csv', index=False)

In [5]:
df_ravdess["emotion"].value_counts()

,count
emotion,
surprised,384
calm,384
fearful,384
happy,384
sad,384
disgust,384
angry,384
neutral,192


In [6]:
import librosa
import numpy as np
import pandas as pd
import os

def extract_features(df_ravdess,mfcc=13):
  features=[]

  for idx, row in df_ravdess.iterrows():
    try:
      if not os.path.exists(row['path']):
        print(f'File not found: {row["path"]}')
        continue


      y,sr=librosa.load(row['path'],sr=16000)

      if len(y)==0:
        print(f'Empty audio file: {row["path"]}')
        continue
      #mfccs

      mfccs=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=mfcc)
      mfccs_mean=np.mean(mfccs,axis=1)

      #chromagram
      chroma=librosa.feature.chroma_stft(y=y,sr=sr)
      chroma_mean=np.mean(chroma,axis=1) # Corrected 'meam' to 'mean'

      #can use mel-spectrogram

      features.append({
          'mfccs': mfccs_mean,
          'chroma':chroma_mean,
          'emotion':row['emotion']
      })

      print(f"Successfully processsed file {idx+1}: {os.path.basename(row['path'])}")

    except Exception as e:
      print(f"Error Processing file{row['path']}:{str(e)}")
      print(f"Error Type: {type(e).__name__}")
  return features

# Use the correct path for extraction
ravdess_path = './ravdess-emotional-speech-audio' # Define ravdess_path
ravdess_data = create_ravdess_dataset(ravdess_path) # Keep create_ravdess_dataset call as it's needed
df_ravdess = pd.DataFrame(ravdess_data) # Keep DataFrame creation

In [7]:
extracted_features = extract_features(df_ravdess)
display(extracted_features)

Successfully processsed file 1: 03-01-08-02-01-02-18.wav
Successfully processsed file 2: 03-01-02-01-02-01-18.wav
Successfully processsed file 3: 03-01-06-02-01-01-18.wav
Successfully processsed file 4: 03-01-08-01-01-01-18.wav
Successfully processsed file 5: 03-01-02-02-01-01-18.wav
Successfully processsed file 6: 03-01-01-01-02-01-18.wav
Successfully processsed file 7: 03-01-08-01-02-01-18.wav
Successfully processsed file 8: 03-01-03-02-02-01-18.wav
Successfully processsed file 9: 03-01-06-02-02-02-18.wav
Successfully processsed file 10: 03-01-08-02-02-02-18.wav
Successfully processsed file 11: 03-01-03-01-01-02-18.wav
Successfully processsed file 12: 03-01-07-02-02-02-18.wav
Successfully processsed file 13: 03-01-07-02-01-01-18.wav
Successfully processsed file 14: 03-01-04-01-01-02-18.wav
Successfully processsed file 15: 03-01-01-01-02-02-18.wav
Successfully processsed file 16: 03-01-02-02-02-02-18.wav
Successfully processsed file 17: 03-01-08-01-02-02-18.wav
Successfully processsed

[{'mfccs': array([-594.88605  ,   30.944561 ,   -2.6231601,   -1.3527832,
          -10.555928 ,    4.4925265,   -5.04574  ,  -12.403404 ,
           -9.987544 ,   -6.345176 ,   -7.0827565,   -4.6671314,
           -4.0566263], dtype=float32),
  'chroma': array([0.46817464, 0.53752196, 0.49274552, 0.54403144, 0.534058  ,
         0.48924   , 0.50269514, 0.5143938 , 0.49982968, 0.5006263 ,
         0.54652953, 0.51520836], dtype=float32),
  'emotion': 'surprised'},
 {'mfccs': array([-677.97925  ,   49.010475 ,    3.7774358,    6.576298 ,
           -2.8550436,    0.7290306,  -12.198942 ,   -8.146392 ,
           -6.3982353,   -4.308462 ,   -6.349617 ,   -4.9416113,
           -2.258174 ], dtype=float32),
  'chroma': array([0.31917617, 0.35529348, 0.4333248 , 0.46636844, 0.45642653,
         0.53913134, 0.49708316, 0.41450438, 0.46041742, 0.52545756,
         0.49193862, 0.34020773], dtype=float32),
  'emotion': 'calm'},
 {'mfccs': array([-470.68356  ,   17.963606 ,  -28.992327 ,   -5.46

In [8]:
# Flatten the features and create a DataFrame
flattened_features = []
for item in extracted_features:
    features_dict = {'emotion': item['emotion']}
    # Flatten mfccs and add to dictionary
    for i, mfcc_value in enumerate(item['mfccs']):
        features_dict[f'mfcc_{i}'] = mfcc_value
    # Flatten chroma and add to dictionary
    for i, chroma_value in enumerate(item['chroma']):
        features_dict[f'chroma_{i}'] = chroma_value
    flattened_features.append(features_dict)

df_features = pd.DataFrame(flattened_features)

# Save the DataFrame to a CSV file
csv_filename = 'extracted_audio_features.csv'
df_features.to_csv(csv_filename, index=False)

print(f"Extracted features saved to {csv_filename}")

Extracted features saved to extracted_audio_features.csv


In [9]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

!pip install tensorflow

x = np.array(df_features.drop('emotion', axis=1).values.tolist())
y = np.array(df_features.emotion.tolist())


label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

y_categorical = to_categorical(y)

print("Data prepared for model training.")
print("Shape of x:", x.shape)
print("Shape of y:", y.shape)
print("Shape of y_categorical:", y_categorical.shape)

Data prepared for model training.
Shape of x: (2880, 25)
Shape of y: (2880,)
Shape of y_categorical: (2880, 8)


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout

def create_model(input_data,num_classes):
  model=Sequential([
      Dense(256,activation='relu',input_shape=(input_data,)),
      Dropout(0.5),
      Dense(128,activation='relu'),
      Dropout(0.5),
      Dense(num_classes,activation='softmax')
  ])

  model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
  return model

In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input # Import Input layer

# Re-define create_model to be accessible in this cell
def create_model(input_dim, num_classes):
  model=Sequential([
      Input(shape=(input_dim,)), # Use Input layer as the first layer
      Dense(256,activation='relu'), # Remove input_shape from Dense layer
      Dropout(0.5),
      Dense(128,activation='relu'),
      Dropout(0.5),
      Dense(64, activation='relu'), # Added another Dense layer
      Dropout(0.5), # Added Dropout for the new layer
      Dense(num_classes,activation='softmax')
  ])

  model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
  return model

# Add the code to load and prepare the data here
# Assuming 'extracted_audio_features.csv' is available
df_features = pd.read_csv('extracted_audio_features.csv')

# Separate features (X) and labels (y)
X = np.array(df_features.drop('emotion', axis=1))
y = np.array(df_features['emotion'])

# Encode the emotion labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Define the number of classes
num_classes = len(label_encoder.classes_)


# Define the number of folds for K-Fold cross-validation
num_folds = 5
skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
accuracy_scores = []
loss_scores = []

print(f"Starting {num_folds}-Fold Cross-Validation with Feature Scaling...")

# --- K-Fold Loop with SCALING ---
for fold, (train_index, test_index) in enumerate(skf.split(X, y_encoded)): # Use y_encoded for stratified splitting
    print(f"--- Starting Fold {fold+1}/{num_folds} ---")

    # 1. Split data for this fold
    X_train, X_test = X[train_index], X[test_index]
    y_train_encoded, y_test_encoded = y_encoded[train_index], y_encoded[test_index]


    # 2. Apply Feature Scaling
    scaler = StandardScaler()
    # Fit the scaler ONLY on the training data
    X_train_scaled = scaler.fit_transform(X_train)
    # Apply the SAME scaling to the test data
    X_test_scaled = scaler.transform(X_test)

    # 3. One-hot encode labels
    y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
    y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

    # 4. Create and train the model using the SCALED data
    model = create_model(X_train_scaled.shape[1], num_classes) # Use scaled data shape for input_dim
    history = model.fit(X_train_scaled, y_train_categorical,
                        epochs=50, # You can adjust the number of epochs per fold
                        batch_size=32,
                        verbose=0) # Set verbose to 0 to reduce output during training

    # 5. Evaluate the model on the SCALED test data
    loss, accuracy = model.evaluate(X_test_scaled, y_test_categorical, verbose=0)

    print(f"Accuracy for Fold {fold+1}: {accuracy*100:.2f}%")
    print(f"Loss for Fold {fold+1}: {loss:.4f}")


    # 6. Save results
    accuracy_scores.append(accuracy)
    loss_scores.append(loss)

# Calculate and print the average performance across all folds
average_accuracy = np.mean(accuracy_scores)
average_loss = np.mean(loss_scores)
std_accuracy = np.std(accuracy_scores)


print("\n--- K-Fold Cross-Validation Results with Feature Scaling ---")
print(f"Average Accuracy: {average_accuracy*100:.2f}%")
print(f"Standard Deviation of Accuracy: {std_accuracy*100:.2f}%")
print(f"Average Loss: {average_loss:.4f}")

Starting 5-Fold Cross-Validation with Feature Scaling...
--- Starting Fold 1/5 ---
Accuracy for Fold 1: 69.62%
Loss for Fold 1: 0.8898
--- Starting Fold 2/5 ---
Accuracy for Fold 2: 68.58%
Loss for Fold 2: 0.8609
--- Starting Fold 3/5 ---
Accuracy for Fold 3: 67.53%
Loss for Fold 3: 0.9120
--- Starting Fold 4/5 ---
Accuracy for Fold 4: 71.01%
Loss for Fold 4: 0.8250
--- Starting Fold 5/5 ---
Accuracy for Fold 5: 74.31%
Loss for Fold 5: 0.7829

--- K-Fold Cross-Validation Results with Feature Scaling ---
Average Accuracy: 70.21%
Standard Deviation of Accuracy: 2.35%
Average Loss: 0.8541
